# LangChain — Advanced Practical Patterns

Follow-up to *LangChain 101*. That notebook covered the basics (models, prompts, chains, memory,
tools, agents). This one assumes all of that and goes one level deeper into patterns
you actually need once a prototype has to survive contact with real usage:

1. Composable runnables — `RunnableParallel`, `RunnableLambda`, `RunnableBranch`
2. Streaming & batching
3. Retries & fallbacks (robustness)
4. Callbacks — custom tracing / token & cost logging
5. Advanced structured output — nested schemas, parallel tool calls
6. Real text splitting for RAG


**Model:** `gemini-3.6-flash` via `langchain-google-genai`, same as before.


In [ ]:
%pip install -q -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-classic \
    langchain-google-genai \
    langchain-text-splitters \
    langchain-huggingface \
    pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests=

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")


Enter your Google AI API key: ··········


## 1 · Composable runnables

`|` is enough for a straight line. Real pipelines branch and fan out. Three building blocks cover
almost every shape you'll need:

- **`RunnableLambda`** — wrap any plain Python function so it can sit in a chain
- **`RunnableParallel`** — run several branches on the *same* input at once, collect a dict of results
- **`RunnableBranch`** — route to a different chain depending on a condition


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

summarize_prompt = ChatPromptTemplate.from_template("Summarize in one sentence:\n\n{text}")
keywords_prompt = ChatPromptTemplate.from_template("List 3 keywords, comma-separated:\n\n{text}")

summarize_chain = summarize_prompt | llm | StrOutputParser()
keywords_chain = keywords_prompt | llm | StrOutputParser()

# Run both branches on the same input in parallel, get a dict back
analysis_chain = RunnableParallel(
    summary=summarize_chain,
    keywords=keywords_chain,
    word_count=RunnableLambda(lambda x: len(x["text"].split())),
)

result = analysis_chain.invoke({
    "text": "LangChain is a framework for building applications powered by language models. "
            "It provides standard interfaces for models, prompts, memory, and tools."
})
print(result)


{'summary': 'LangChain is a framework that facilitates the development of language model-powered applications by providing standard interfaces for components like models, prompts, memory, and tools.', 'keywords': 'LangChain, framework, language models', 'word_count': 21}


In [ ]:
from langchain_core.runnables import RunnableBranch

# Route short vs. long text to different handling — no if/else glue code outside the chain
short_chain = RunnableLambda(lambda x: f"(short text) {x['text']}")
long_chain = summarize_chain

router = RunnableBranch(
    (lambda x: len(x["text"].split()) < 20, short_chain),
    long_chain,  # default branch
)

print(router.invoke({"text": "Too short to summarize."}))
print(router.invoke({"text": result["summary"] * 3}))  # long enough to trigger summarization


(short text) Too short to summarize.
LangChain is a development framework that simplifies the creation of language model applications by providing standardized interfaces for essential components like models, prompts, memory, and tools.


## 2 · Streaming & batching

- **`.stream()`** — get tokens as they're generated, essential for responsive UIs
- **`.batch()`** — send multiple independent inputs concurrently instead of looping `.invoke()`


In [ ]:
for chunk in llm.stream("Write a 2-line poem about debugging code."):
    print(chunk.content, end="", flush=True)


[{'type': 'text', 'text': 'I chased a silent phantom through the code all night,\n', 'index': 0}][{'type': 'text', 'text': 'To find a single semicolon made the system right.', 'index': 0}][{'type': 'text', 'text': '', 'extras': {'signature': 'EowbCokbARFNMg+4ZE3WNsy3VnvTuRxST9fS8d0tc8dinBJjdXpBDuwMnIxwhdifHgf64J6B29uqyX6lly4cghJWMVpxtzB2Gp9i0CNd8iySP5Ruj546nR/RU+YRdHjVZWjsaba4wxixBxlAvvDvUEhImXLm9Psn9DVPNB7Hmc3L1IC58tHNzLGEaO4WptePlpIcF3RDUaPHNuv310aBLOKLB7PabgTw8wvAZUOnDMKhzOdLZsQkK3AsIcqsvQWfBpBwekumxTwnwRqpnfwfQKv4O+o2YXWErbL4e7exndeWOY1FMyC7StAf4Uc3dvaO3Fe1sgyS3bWVphaSo++a2KPWb3R4R4/Ww/OOIiKtY06CXp0FdtrY8Jw1Zo5Fv5ZWX7BhJhwjv/leawYniSiAck8tgE9edSuqiMXjI/4nP5fjO2pvExTGBnE1vF/hgnH3WUQ0mI5PyDagPLd0jwkG5T1be6dSpU5fdqmdD02lAfd479bvIV7aqPNxHqifVIhEH7cK8MOiEZ+r6k7qFniD9+npHAAX40O6OovxTkNeMOH+YRvVqCCbTESZqAJv8i0KcAsjBLAchpKg++W3V1ReOVbI6lX7fUc7tE9yq64UzSQVjnGV4GO5tXKpXd+J139WrKpI/6bRtcfdaU6KSV48swVxE46JFB6+oQ9Jhc6ZDbgesgllq2viOYkeILnCS013me1G0yWydNtOSmHecqSZg+f6AOUwG0vt0vnZsqhqZ/oI9R+NhOHoz

In [ ]:
questions = [
    "What is LCEL?",
    "What is a vector store?",
    "What is a tool call?",
]

# .batch() fires these concurrently — much faster than a Python for-loop of .invoke()
answers = llm.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q}\nA: {a.content[:120]}...\n")


Q: What is LCEL?
A: [{'type': 'text', 'text': '**LCEL** stands for **LangChain Expression Language**. It is a declarative language designed by LangChain to make it easy to compose and chain different components together to build Large Language Model (LLM) applications.\n\nThink of LCEL as a way to construct "pipelines" for your data, where the output of one component becomes the input of the next.\n\n---\n\n### The Core Concept: The Pipe Operator (`|`)\n\nLCEL heavily relies on the **pipe operator (`|`)**, similar to how Unix command-line pipes work. \n\nIn standard Python, chaining components might look like this:\n`output = parse(model(prompt(input)))` (which can get messy and hard to read).\n\nIn LCEL, you write it sequentially from left to right:\n```python\nchain = prompt | model | parser\n```\n\n### A Simple Example\n\nHere is a basic example of how LCEL is used in Python:\n\n```python\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_core.output_parsers impo

## 3 · Retries & fallbacks

Two different failure modes, two different tools:

- **`.with_retry()`** — same model, try again on transient errors (rate limits, timeouts)
- **`.with_fallbacks([...])`** — if the *first* model fails entirely (or you want a cheaper/faster
  backup), fall through to another model


In [ ]:
robust_llm = llm.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True,
)

# Same interface as llm — just more resilient to transient failures
print(robust_llm.invoke("Say hello in French.").content)


[{'type': 'text', 'text': 'Bonjour !', 'extras': {'signature': 'EqMECqAEARFNMg8RmH23umNqPDIlsJPUJ5pkBubkBSJZupH2lC/aNQNk64gHxk9dIBwlPjcKeVsp1+wlq4K+cEas7SiFEQVZsidx1uToPaxU+PamqNIVmuDVsLzhx3mg8vpjiRa3/PWnYCH/lGKFv28hED10soIj7AJ+7b7VjwwGkut67niAqayeOPWoqIY4ZX4rKFW38FHuU7VPpLJUcSoC/HFrgNIPdEnlpb56yP+2/xMDlaVcJzL9PYUL7lEPS4QL+Rh0oaCQctBpI7qtX3rTHODBueMm0psmfQaUcjj4+L8B6WwtAUzTcXFd83Wy6yhlh3JULysaoFTWjIcLRAx9UsQxoiemFdACGm8SxMhB8DM9tvYGxm7RvuKx9nULVg1hGU2XanS4xjZQhwuI+r/NYm07bshg8chtWc+QftgmM5kGvMJjr+2rsBd0ucqEGalAPaA3Iq/LbGKx3GKIntEyI6dci1HFgvSszY1cEQcF8upwfWZlFFLt3rekpUkVwWsSfKwZ/yql9BZdllo3Ti0tQwjxbmvUaDnk6cqm43vi3nO8w/JzKRBCc6yPx9hS4zMkt7/PIBaFXP/iIZ3EfYYqM+96a9fwPd7gbdilDWc6G5p8S+piyQsfPEk4tbWgpVK66ozOSheiR4yfVyHrG0cxob2OtkVCJtWifI7xbh1zZNsp/IzItqi8mUd8nSzKrscehERtijwwdPJDP37zHP9gIWNc4w=='}}]


In [ ]:
backup_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash-lite", temperature=0.3)

resilient_llm = llm.with_fallbacks([backup_llm])

# If the primary model call raises, resilient_llm transparently retries with backup_llm
print(resilient_llm.invoke("Say hello in Spanish.").content)


GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 3.383458043s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'q

In [ ]:
print(resilient_llm.invoke("Say hello in Spanish.").response_metadata)

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 14.630470348s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [ ]:
from google.api_core.exceptions import ResourceExhausted

backup_llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash-lite",
    temperature=0.3
)

resilient_llm = llm.with_fallbacks(
    [backup_llm],
    exceptions_to_handle=(ResourceExhausted,)
)

response = resilient_llm.invoke("Say hello in Spanish.")

print(response.content)

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 32.16990027s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'q

In [ ]:
from langchain_google_genai.chat_models import GoogleRateLimitError

In [ ]:
backup_llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    # temperature=0.3
)

resilient_llm = llm.with_fallbacks(
    [backup_llm],
    exceptions_to_handle=(GoogleRateLimitError,)
)

response = resilient_llm.invoke(
    "Say hello in Spanish."
)

print(response.response_metadata)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}


## 4 · Callbacks — custom tracing & cost tracking

Callbacks hook into every stage of execution (LLM start/end, tool start/end, chain start/end)
without touching your chain's logic — useful for logging, cost tracking, or a custom dashboard.


In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounterHandler(BaseCallbackHandler):
    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.calls = 0

    def on_llm_end(self, response, **kwargs):
        self.calls += 1
        for generation in response.generations:
            for gen in generation:
                usage = getattr(gen.message, "usage_metadata", None) if hasattr(gen, "message") else None
                if usage:
                    self.total_input_tokens += usage.get("input_tokens", 0)
                    self.total_output_tokens += usage.get("output_tokens", 0)


counter = TokenCounterHandler()

llm.invoke("What is the capital of Japan?", config={"callbacks": [counter]})
llm.invoke("What is the capital of France?", config={"callbacks": [counter]})

print(f"Calls: {counter.calls}")
print(f"Input tokens: {counter.total_input_tokens}, Output tokens: {counter.total_output_tokens}")


Calls: 2
Input tokens: 16, Output tokens: 161


This is the same mechanism LangSmith's tracing is built on — a callback handler that ships
traces to a dashboard instead of a Python counter. Worth reaching for once you need to debug a
chain in production rather than in a notebook.


## 5 · Advanced structured output

Real schemas are rarely flat. Nested models, optional fields, and lists of typed objects all work
the same way — and a model can also request **multiple tool calls in parallel** in a single turn.


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class LineItem(BaseModel):
    item: str
    quantity: int
    unit_price: float

class Invoice(BaseModel):
    vendor: str
    items: list[LineItem]
    tax_rate: Optional[float] = Field(default=None, description="As a decimal, e.g. 0.08")
    total: float


structured_llm = llm.with_structured_output(Invoice)

result = structured_llm.invoke("""
Extract the invoice details:
Vendor: Acme Office Supplies
- 10 notebooks at $3.50 each
- 2 staplers at $12.00 each
Tax rate: 8%
Total: $75.20
""")
print(result)
print(result.items[0].item, result.items[0].quantity)


vendor='Acme Office Supplies' items=[LineItem(item='notebooks', quantity=10, unit_price=3.5), LineItem(item='staplers', quantity=2, unit_price=12.0)] tax_rate=0.08 total=75.2
notebooks 10


In [ ]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a ticker symbol."""
    fake_prices = {"AAPL": "$227.50", "GOOG": "$185.20"}
    return fake_prices.get(ticker.upper(), "Unknown ticker")

llm_with_tools = llm.bind_tools([get_stock_price])

# Ask for two lookups at once — a capable model returns two tool_calls in one response
resp = llm_with_tools.invoke("What are the prices of AAPL and GOOG?")
for call in resp.tool_calls:
    print(call["name"], call["args"])


get_stock_price {'ticker': 'AAPL'}
get_stock_price {'ticker': 'GOOG'}


## 6 · Real text splitting for RAG

The earlier RAG demo embedded three hand-written sentences. Real documents need to be split into
overlapping chunks first — too large and retrieval gets imprecise, too small and you lose context.
`RecursiveCharacterTextSplitter` splits on paragraph/sentence boundaries where possible.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

document = """
LangChain is a framework for developing applications powered by language models. It began in
October 2022 and has since grown a large ecosystem of integrations across model providers, vector
stores, and tools.

LCEL, the LangChain Expression Language, lets developers compose chains declaratively using the
pipe operator. A chain built with LCEL automatically supports streaming, batching, and async
execution without any extra code.

LangGraph extends these ideas with an explicit graph of nodes and edges, giving developers control
over branching, cycles, and persisted state — useful for agents that need more structure than a
simple loop.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
)

chunks = splitter.split_text(document)
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c.strip())
    print()


--- chunk 0 (209 chars) ---
LangChain is a framework for developing applications powered by language models. It began in
October 2022 and has since grown a large ecosystem of integrations across model providers, vector
stores, and tools.

--- chunk 1 (188 chars) ---
LCEL, the LangChain Expression Language, lets developers compose chains declaratively using the
pipe operator. A chain built with LCEL automatically supports streaming, batching, and async

--- chunk 2 (33 chars) ---
execution without any extra code.

--- chunk 3 (207 chars) ---
LangGraph extends these ideas with an explicit graph of nodes and edges, giving developers control
over branching, cycles, and persisted state — useful for agents that need more structure than a
simple loop.



## 7 · Advanced retrieval — multi-query + contextual compression

Two upgrades over a plain similarity-search retriever:

- **Multi-query retrieval** — the LLM rewrites your question several different ways, retrieves for
  each, and merges results. Catches relevant chunks that don't share the question's exact wording.
- **Contextual compression** — after retrieving, an LLM strips each chunk down to only the parts
  relevant to the question, so the final prompt isn't padded with noise.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_texts([c.strip() for c in chunks])

base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})


In [ ]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=backup_llm,
)

docs = multi_query_retriever.invoke("When did the LangChain project start and what came after?")
for d in docs:
    print("-", d.page_content[:90].replace("\n", " "), "...")


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


- LangChain is a framework for developing applications powered by language models. It began  ...
- LCEL, the LangChain Expression Language, lets developers compose chains declaratively usin ...
- LangGraph extends these ideas with an explicit graph of nodes and edges, giving developers ...


In [ ]:
print(multi_query_retriever)

retriever=VectorStoreRetriever(tags=['InMemoryVectorStore', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x7e2ea9f50f50>, search_kwargs={'k': 3}) llm_chain=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language model assistant. Your task is\n    to generate 3 different versions of the given user\n    question to retrieve relevant documents from a vector  database.\n    By generating multiple perspectives on the user question,\n    your goal is to help the user overcome some of the limitations\n    of distance-based similarity search. Provide these alternative\n    questions separated by newlines. Original question: {question}')
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.3.7'}}, profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21',

In [ ]:
question = "When did the LangChain project start and what came after?"

queries = multi_query_retriever.llm_chain.invoke({
    "question": question
})

print("Generated queries:")
print(queries)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated queries:
['What is the launch date of the LangChain framework, and how has the project evolved since its initial release?', 'In what year was LangChain created, and what new products or updates were introduced following its inception?', "Can you provide the timeline of LangChain's inception along with the major developments that took place in its ecosystem afterward?"]


                              DOCUMENT
                                 |
                                 ▼
                    RecursiveCharacterTextSplitter
                                 |
                                 ▼
                           chunks of text
                                 |
                                 ▼
                       Gemini Embedding Model
                                 |
                                 ▼
                            Vector Store
                                 |
                                 ▼
                          base_retriever
                                 |
                  USER QUESTION ─┤
                                 |
                                 ▼
                       MultiQueryRetriever
                                 |
                                 ▼
                       LLM generates
                       multiple queries
                         /       |       \
                        /        |        \
                       ▼         ▼         ▼
                    Search     Search     Search
                       \         |         /
                        \        |        /
                         ▼       ▼       ▼
                       Combine
                     + deduplicate
                           |
                           ▼
                    Retrieved Documents

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

compressed_docs = compression_retriever.invoke(
    "What does LCEL give you for free?"
)

for d in compressed_docs:
    print("-", d.page_content)

- A chain built with LCEL automatically supports streaming, batching, and async
- execution without any extra code.


In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# 1. Create the compressor
compressor = LLMChainExtractor.from_llm(llm)

# 2. Create compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

# 3. Your question
query = "What does LCEL give you for free?"

# --------------------------------------------------
# STEP 1: See what the normal retriever retrieves
# --------------------------------------------------

retrieved_docs = base_retriever.invoke(query)

print("=" * 80)
print("📚 RETRIEVED DOCUMENTS")
print("=" * 80)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Document {i} ---")
    print(doc.page_content)


# --------------------------------------------------
# STEP 2: Compress the retrieved documents
# --------------------------------------------------

compressed_docs = compression_retriever.invoke(query)

print("\n" + "=" * 80)
print("✂️ COMPRESSED DOCUMENTS")
print("=" * 80)

for i, doc in enumerate(compressed_docs, 1):
    print(f"\n--- Compressed Document {i} ---")
    print(doc.page_content)

📚 RETRIEVED DOCUMENTS

--- Retrieved Document 1 ---
LCEL, the LangChain Expression Language, lets developers compose chains declaratively using the
pipe operator. A chain built with LCEL automatically supports streaming, batching, and async

--- Retrieved Document 2 ---
LangChain is a framework for developing applications powered by language models. It began in
October 2022 and has since grown a large ecosystem of integrations across model providers, vector
stores, and tools.

--- Retrieved Document 3 ---
execution without any extra code.

✂️ COMPRESSED DOCUMENTS

--- Compressed Document 1 ---
streaming, batching, and async

--- Compressed Document 2 ---
execution without any extra code.


In [ ]:
print("\n" + "=" * 80)
print("📊 COMPRESSION SUMMARY")
print("=" * 80)

retrieved_chars = sum(len(doc.page_content) for doc in retrieved_docs)
compressed_chars = sum(len(doc.page_content) for doc in compressed_docs)

print(f"Retrieved documents : {len(retrieved_docs)}")
print(f"Compressed documents: {len(compressed_docs)}")
print(f"Retrieved characters: {retrieved_chars}")
print(f"Compressed characters: {compressed_chars}")

if retrieved_chars:
    reduction = (1 - compressed_chars / retrieved_chars) * 100
    print(f"Reduction           : {reduction:.2f}%")


📊 COMPRESSION SUMMARY
Retrieved documents : 3
Compressed documents: 2
Retrieved characters: 430
Compressed characters: 63
Reduction           : 85.35%


## 8 · Conversational RAG

A follow-up question like *"what about the overlap?"* only makes sense with chat history. A
**history-aware retriever** first rewrites the question into a standalone one using the
conversation so far, *then* retrieves — so retrieval quality doesn't degrade turn over turn.


In [ ]:
from langchain_classic.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)

# --------------------------------------------------
# 1. Prompt for rewriting follow-up questions
# --------------------------------------------------

contextualize_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Given the chat history and a follow-up question, "
        "rewrite it as a standalone question."
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


# --------------------------------------------------
# 2. History-aware retriever
# --------------------------------------------------

history_aware_retriever = create_history_aware_retriever(
    backup_llm,
    base_retriever,
    contextualize_prompt,
)


# --------------------------------------------------
# 3. Prompt for generating the final answer
# --------------------------------------------------

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only this context:\n\n{context}"
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


# --------------------------------------------------
# 4. Document processing chain
# --------------------------------------------------

document_chain = create_stuff_documents_chain(
    backup_llm,
    answer_prompt,
)


# --------------------------------------------------
# 5. Complete conversational RAG chain
# --------------------------------------------------

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    document_chain,
)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

r1 = conversational_rag_chain.invoke({"input": "What is LCEL?", "chat_history": chat_history})
print("A1:", r1["answer"])
chat_history += [HumanMessage(r1["input"] if False else "What is LCEL?"), AIMessage(r1["answer"])]

# Follow-up only makes sense with history — "it" refers to LCEL
r2 = conversational_rag_chain.invoke({"input": "Does it support streaming?", "chat_history": chat_history})
print("A2:", r2["answer"])


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A1: Based on the provided context, LCEL stands for the LangChain Expression Language. It allows developers to compose chains declaratively using the pipe operator. Chains built using LCEL automatically support streaming, batching, and async execution without requiring extra code.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleGenerativeAIError: Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

## 9 · Production-grade custom tools

Real tools need input validation and graceful error handling — a model can pass malformed
arguments, or the underlying call can fail. `StructuredTool` with an explicit `args_schema` gives
you both.


In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(description="City name, e.g. 'Tokyo'")
    units: str = Field(default="celsius", description="'celsius' or 'fahrenheit'")


def _get_weather(city: str, units: str = "celsius") -> str:
    fake_data = {"tokyo": 18, "cairo": 22, "london": 14}
    key = city.strip().lower()
    if key not in fake_data:
        # Return an error string rather than raising — lets the model recover gracefully
        return f"Error: no weather data for '{city}'. Try a major city name."
    temp_c = fake_data[key]
    if units == "fahrenheit":
        return f"{city}: {temp_c * 9 / 5 + 32:.0f}F"
    return f"{city}: {temp_c}C"


weather_tool = StructuredTool.from_function(
    func=_get_weather,
    name="get_weather",
    description="Get current weather for a city, optionally in fahrenheit.",
    args_schema=WeatherInput,
)

llm_with_tool = llm.bind_tools([weather_tool])
resp = llm_with_tool.invoke("What's the weather in Cairo in fahrenheit?")
for call in resp.tool_calls:
    print(call["name"], call["args"], "->", weather_tool.invoke(call["args"]))


## 10 · Caching

Identical prompts don't need to hit the API twice. An LLM cache stores the result of each exact
prompt+model combination — instant on a repeat, and free.


In [ ]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
import time

set_llm_cache(InMemoryCache())

start = time.time()
backup_llm.invoke("What is the capital of Australia?")
print(f"First call: {time.time() - start:.2f}s")

start = time.time()
backup_llm.invoke("What is the capital of AUS?")  # identical prompt -> served from cache
print(f"Second call (cached): {time.time() - start:.2f}s")


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


First call: 44.81s
Second call (cached): 0.00s


In [ ]:
import time
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

set_llm_cache(InMemoryCache())

# First
start = time.time()
backup_llm.invoke("What is the capital of Australia?")
print(f"1: {time.time() - start:.2f}s")

# Exact same prompt → cache hit
start = time.time()
backup_llm.invoke("What is the capital of Australia?")
print(f"2: {time.time() - start:.2f}s")

# Slightly different → cache miss
start = time.time()
backup_llm.invoke("What is the capital of AUS?")
print(f"3: {time.time() - start:.2f}s")

# Exact same "AUS" prompt → cache hit
start = time.time()
backup_llm.invoke("What is the capital of AUS?")
print(f"4: {time.time() - start:.2f}s")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1: 71.28s
2: 0.00s


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


3: 10.31s
4: 0.00s


For anything beyond a single notebook session, swap `InMemoryCache()` for
`SQLiteCache(database_path=".langchain.db")` so the cache survives a restart.


## 11 · Putting it together

One chain that reflects what a small production feature actually looks like: a cached, retrying,
tool-using model that returns structured output.


In [ ]:
production_llm = (
    llm
    .with_retry(stop_after_attempt=3)
    .with_fallbacks([backup_llm])
    .bind_tools([weather_tool])
)

resp = production_llm.invoke(
    "Is it warmer in Tokyo or Cairo right now?",
    config={"callbacks": [counter]},
)
print(resp.tool_calls or resp.content)
print(f"\nRunning total — calls: {counter.calls}, "
      f"input tokens: {counter.total_input_tokens}, output tokens: {counter.total_output_tokens}")


## Recap

| # | Concept | Why it matters |
|---|---|---|
| 1 | RunnableParallel / Lambda / Branch | Fan-out, custom logic, and routing inside a chain |
| 2 | Streaming & batching | Responsive UIs, faster bulk processing |
| 3 | Retries & fallbacks | Survive transient failures and provider outages |
| 4 | Callbacks | Tracing, cost tracking, custom logging |
| 5 | Advanced structured output | Nested schemas, parallel tool calls |
| 6 | Text splitting | Chunking real documents correctly |
| 7 | Multi-query + compression retrieval | Better recall, less noise in context |
| 8 | Conversational RAG | Retrieval that understands follow-up questions |
| 9 | StructuredTool | Validated inputs, graceful tool errors |
| 10 | Caching | Skip duplicate calls — faster and cheaper |
| 11 | Combined chain | What these patterns look like stacked together |

**Natural next step:** LangGraph, for when a single chain isn't enough structure — explicit state,
branching, cycles, multi-agent handoffs, and human-in-the-loop checkpoints.


In [ ]:
!pip install -q langchain-google-genai langchain-huggingface sentence-transformers

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

# --------------------------------------------------
# 1. LLM
# --------------------------------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

# --------------------------------------------------
# 2. Embedding model
# --------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import numpy as np

cache = []

SIMILARITY_THRESHOLD = 0.80


def semantic_cache_invoke(prompt):

    # Convert prompt to embedding
    query_embedding = np.array(
        embeddings.embed_query(prompt)
    )

    # ---------------------------------------------
    # Check existing cache
    # ---------------------------------------------

    for cached_prompt, cached_embedding, cached_response in cache:

        similarity = np.dot(
            query_embedding,
            cached_embedding
        ) / (
            np.linalg.norm(query_embedding)
            * np.linalg.norm(cached_embedding)
        )

        print(
            f"Similarity with '{cached_prompt}': "
            f"{similarity:.3f}"
        )

        if similarity >= SIMILARITY_THRESHOLD:
            print("✅ SEMANTIC CACHE HIT")
            return cached_response

    # ---------------------------------------------
    # Cache miss → call LLM
    # ---------------------------------------------

    print("❌ CACHE MISS → Calling Gemini")

    response = llm.invoke(prompt)

    # Store
    cache.append(
        (
            prompt,
            query_embedding,
            response
        )
    )

    return response

In [ ]:
r1 = semantic_cache_invoke(
    "What is the capital of Australia?"
)

print(r1.content)

❌ CACHE MISS → Calling Gemini
[{'type': 'text', 'text': 'The capital of Australia is **Canberra**.', 'extras': {'signature': 'EoEFCv4EARFNMg/ubG/jaqUZXXL5m/JbjkkwUCWwTBxD0XZd4PtREwA/MxsXy/66jKcBg4l6IhSVETe+kpH03/VAswqktNy/dUxWO7KmkJjjQku4FurlpZRqpWKSbNxf6y1ZC6OqIwR57xLv6d2O8bN0YGb9kjaJYKHo51Gzs764xP66dBzn2PONCP3VkFUa7ECBn3HCMPtJtmbr1K95P6mEs9ECa8v59rUNYzKLd9uIYxf6zTPaiRFfp6xauRrMkKJPrzuExzS7M1/BqnRKFIAWk+guEWUTItpFMsNNCHrdW/l/143T5nQMcYEncj3BP6S1pa9G6fuaz+dtkuF55qPKE7I4aefNUX0FEfXKlK1E2kSqW0fsGUP9iNh4D3RHICyEEyUNN+39UljmORVNBAC8j1pFkp5GuIHVteHIg0bWl8hnnCIn2EQw/msDGp2KCV6e49f+l8zrNHLqwGwD0BsGVNxbBCIdwk9tJXpzJkWWs7i4gOHg2dpOY7oav3h/DAKMKECg5T+q21NMN08WfxCmT/ZOVOI24IbephM7GZFTjXVgS09zKUCxjl/qscumtDhVbyRulD42lZDk/aF5l6YsNjUI+v8ttVqLrhwykfO/2UWgJkioHqBM+JMo5EEhrPjh5CR0O3CneEkaZOlONU/m3f8GvEO0QpAH2m51cTZhjTXR1gquXK11YiaHNa0eLVMA6yMlKAxmBOprqVjy9oV/g73JQabpRYxRfbbccnxXwZvaJBPwLNqHcVvXaoaH0pIkUl+op8qB7NPN2jkepkmwLoYcQLR5Vk7QEbIpSt4hV5wXsJhY03YCG1CYUTwUiE6IK3n0Fa0/bsLqrpzyv8V6Xno='}}]


In [ ]:
r2 = semantic_cache_invoke(
    "What's Australia's capital city?"
)

print(r2.content)

Similarity with 'What is the capital of Australia?': 0.938
✅ SEMANTIC CACHE HIT
[{'type': 'text', 'text': 'The capital of Australia is **Canberra**.', 'extras': {'signature': 'EoEFCv4EARFNMg/ubG/jaqUZXXL5m/JbjkkwUCWwTBxD0XZd4PtREwA/MxsXy/66jKcBg4l6IhSVETe+kpH03/VAswqktNy/dUxWO7KmkJjjQku4FurlpZRqpWKSbNxf6y1ZC6OqIwR57xLv6d2O8bN0YGb9kjaJYKHo51Gzs764xP66dBzn2PONCP3VkFUa7ECBn3HCMPtJtmbr1K95P6mEs9ECa8v59rUNYzKLd9uIYxf6zTPaiRFfp6xauRrMkKJPrzuExzS7M1/BqnRKFIAWk+guEWUTItpFMsNNCHrdW/l/143T5nQMcYEncj3BP6S1pa9G6fuaz+dtkuF55qPKE7I4aefNUX0FEfXKlK1E2kSqW0fsGUP9iNh4D3RHICyEEyUNN+39UljmORVNBAC8j1pFkp5GuIHVteHIg0bWl8hnnCIn2EQw/msDGp2KCV6e49f+l8zrNHLqwGwD0BsGVNxbBCIdwk9tJXpzJkWWs7i4gOHg2dpOY7oav3h/DAKMKECg5T+q21NMN08WfxCmT/ZOVOI24IbephM7GZFTjXVgS09zKUCxjl/qscumtDhVbyRulD42lZDk/aF5l6YsNjUI+v8ttVqLrhwykfO/2UWgJkioHqBM+JMo5EEhrPjh5CR0O3CneEkaZOlONU/m3f8GvEO0QpAH2m51cTZhjTXR1gquXK11YiaHNa0eLVMA6yMlKAxmBOprqVjy9oV/g73JQabpRYxRfbbccnxXwZvaJBPwLNqHcVvXaoaH0pIkUl+op8qB7NPN2jkepkmwLoYcQLR5Vk7QEbIpSt4hV5wXsJhY03YC

In [ ]:
r3 = semantic_cache_invoke(
    "Which city is the capital of Australia?"
)

print(r3.content)

Similarity with 'What is the capital of Australia?': 0.933
✅ SEMANTIC CACHE HIT
[{'type': 'text', 'text': 'The capital of Australia is **Canberra**.', 'extras': {'signature': 'EoEFCv4EARFNMg/ubG/jaqUZXXL5m/JbjkkwUCWwTBxD0XZd4PtREwA/MxsXy/66jKcBg4l6IhSVETe+kpH03/VAswqktNy/dUxWO7KmkJjjQku4FurlpZRqpWKSbNxf6y1ZC6OqIwR57xLv6d2O8bN0YGb9kjaJYKHo51Gzs764xP66dBzn2PONCP3VkFUa7ECBn3HCMPtJtmbr1K95P6mEs9ECa8v59rUNYzKLd9uIYxf6zTPaiRFfp6xauRrMkKJPrzuExzS7M1/BqnRKFIAWk+guEWUTItpFMsNNCHrdW/l/143T5nQMcYEncj3BP6S1pa9G6fuaz+dtkuF55qPKE7I4aefNUX0FEfXKlK1E2kSqW0fsGUP9iNh4D3RHICyEEyUNN+39UljmORVNBAC8j1pFkp5GuIHVteHIg0bWl8hnnCIn2EQw/msDGp2KCV6e49f+l8zrNHLqwGwD0BsGVNxbBCIdwk9tJXpzJkWWs7i4gOHg2dpOY7oav3h/DAKMKECg5T+q21NMN08WfxCmT/ZOVOI24IbephM7GZFTjXVgS09zKUCxjl/qscumtDhVbyRulD42lZDk/aF5l6YsNjUI+v8ttVqLrhwykfO/2UWgJkioHqBM+JMo5EEhrPjh5CR0O3CneEkaZOlONU/m3f8GvEO0QpAH2m51cTZhjTXR1gquXK11YiaHNa0eLVMA6yMlKAxmBOprqVjy9oV/g73JQabpRYxRfbbccnxXwZvaJBPwLNqHcVvXaoaH0pIkUl+op8qB7NPN2jkepkmwLoYcQLR5Vk7QEbIpSt4hV5wXsJhY03YC